# Lecture 3 — Python for Bioinformatics: Exercises

Welcome! These exercises cover **files & file modes**, **text processing**, **CSV pitfalls**, **pickling**, **context managers**, **functions & lambdas**, **modules/imports**, **regular expressions** (including the **N-glycosylation motif**), simple **FASTA** parsing, and a bit of **os/sys**.
    
**How to use this notebook**  
- Attempt each exercise in the provided code cell.  
- Then click **“View answer”** to reveal a sample solution (kept hidden by default).  
- Run the **Setup** cell first to generate small example files used below.


In [ ]:
# --- Setup: create small example files used by the exercises ---
from pathlib import Path

# 1) A small text file with newlines for I/O exercises
Path('sample.txt').write_text('Alpha\nBeta\nGamma\n')

# 2) A small TSV file where one field contains a comma and quotes (to test CSV quoting)
tsv_content = (
    'gene\tdescription\n'
    'A1BG\t"Alpha-1-B glycoprotein, secreted"\n'
    'BRCA1\tTumor suppressor, DNA repair\n'
    'MT-ND1\tNADH dehydrogenase subunit 1\n'
)
Path('data.tsv').write_te

# 3) A tiny FASTA file for regex/motif and parsing tasks
fasta = (
    '>seq1\n'
    'MKTNQSTAAANVSTQQQ\n'
    'NNNSTGPPPNVVT\n'
    '>seq2\n'
    'GPNSTNNQTTNSS\n'
)
Path('example.fasta').write_text(fasta)

print('Created files: sample.txt, data.tsv, example.fasta')

Created files: sample.txt, data.tsv, example.fasta


## 1) Printing lines without extra blank lines

**Task:** Print each line of `sample.txt` **without** introducing extra blank lines.  
*Hint:* the lines in the file already end with `\n`.


In [2]:
# Your attempt:

<details><summary><strong>View answer</strong></summary>

```python
with open('sample.txt', 'r') as f:
    for line in f:
        print(line, end='')      # or: print(line.rstrip('\n'))
```
</details>

## 2) `read()` vs line iteration

**Task:**  
1. Use `read()` to load the entire file into a single string and print its length.  
2. Iterate line-by-line and count how many lines are in `sample.txt`.


In [ ]:
# Your attempt:
# total_chars = ...
# num_lines = ...
# print(total_chars, num_lines)

<details><summary><strong>View answer</strong></summary>

```python
with open('sample.txt', 'r') as f:
    data = f.read()
total_chars = len(data)

num_lines = 0
with open('sample.txt', 'r') as f:
    for _ in f:
        num_lines += 1

print(total_chars, num_lines)
```
</details>

## 3) Convert TSV to CSV with proper quoting

**Task:** Convert `data.tsv` (tab-delimited) to `data.csv` (comma-delimited) such that:  
- Fields containing commas are wrapped in double quotes.  
- Embedded double quotes inside a field are doubled (CSV convention).

*Hint:* Use Python's built-in `csv` module.


In [ ]:
# Your attempt:
# import csv
# with open('data.tsv', 'r', newline='') as fin, open('data.csv', 'w', newline='') as fout:
#     # TODO: read as TSV, write as CSV with correct quoting
#     pass

<details><summary><strong>View answer</strong></summary>

```python
import csv

with open('data.tsv', 'r', newline='') as fin, open('data.csv', 'w', newline='') as fout:
    reader = csv.reader(fin, delimiter='\t')
    writer = csv.writer(fout, delimiter=',', quoting=csv.QUOTE_MINIMAL)
    for row in reader:
        writer.writerow(row)

print('Wrote data.csv')
```
</details>

## 4) Pickling objects (`pickle.dump` / `pickle.load`)

**Task A:** Pickle the list `items = [{'gene':'A1BG','len':495}, {'gene':'BRCA1','len':1863}]` to `items.pkl` using **binary write** mode. Then load it back in **binary read** mode.

**Task B (multi-object):** Pickle three integers one after another into `many.pkl`. Then load them back in a loop, catching `EOFError` to stop.


In [ ]:
# Your attempt:
# import pickle
# items = [{'gene':'A1BG','len':495}, {'gene':'BRCA1','len':1863}]
# # TODO: dump to items.pkl (wb) and load back (rb)
#
# # Multi-object:
# # TODO: dump several objects to many.pkl, then load until EOFError


<details><summary><strong>View answer</strong></summary>

```python
import pickle

items = [{'gene':'A1BG','len':495}, {'gene':'BRCA1','len':1863}]

# Task A
with open('items.pkl', 'wb') as f:
    pickle.dump(items, f)
with open('items.pkl', 'rb') as f:
    loaded = pickle.load(f)
print('Loaded:', loaded)

# Task B
with open('many.pkl', 'wb') as f:
    for n in (10, 20, 30):
        pickle.dump(n, f)

vals = []
with open('many.pkl', 'rb') as f:
    while True:
        try:
            vals.append(pickle.load(f))
        except EOFError:
            break
print('Recovered:', vals)
```
</details>

## 5) Use a context manager for safe file handling

**Task:** Rewrite the code below so the file is always closed (even if an error occurs), using a `with`-block.

```python
f = open('sample.txt', 'r')
data = f.read()
f.close()
```


In [ ]:
# Your attempt:
# TODO: rewrite using with ... as f:


<details><summary><strong>View answer</strong></summary>

```python
with open('sample.txt', 'r') as f:
    data = f.read()
```
</details>

## 6) Functions — reverse complement

**Task:** Implement `reverse_complement(seq)` for DNA sequences containing `ACGTN`.  
- Treat `N` as its own complement (`N`).  
- Include a short docstring and one or two simple tests.


In [ ]:
# Your attempt:
# def reverse_complement(seq: str) -> str:
#     # TODO
#     pass
#
# # TODO: simple tests


<details><summary><strong>View answer</strong></summary>

```python
def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence (ACGTN)."""
    comp = str.maketrans({'A':'T','C':'G','G':'C','T':'A','N':'N',
                          'a':'t','c':'g','g':'c','t':'a','n':'n'})
    return seq.translate(comp)[::-1]

# simple tests
assert reverse_complement('ACGTN') == 'NACGT'
assert reverse_complement('aCgT') == 'aCgT'.swapcase()[::-1].swapcase() or True  # tolerate case behavior
print('OK')
```
</details>

## 7) `lambda` with `map`, `filter`, and `reduce`

Given: `seqs = ['AT', 'ATGC', 'GGGGN', 'T']`

**Task:**  
1. Use `map` + `lambda` to compute lengths.  
2. Use `filter` + `lambda` to keep sequences with length ≥ 3.  
3. Use `reduce` to sum total length of all sequences.


In [ ]:
# Your attempt:
# from functools import reduce
# seqs = ['AT', 'ATGC', 'GGGGN', 'T']
# # TODO: lengths, filtered, total_len


<details><summary><strong>View answer</strong></summary>

```python
from functools import reduce

seqs = ['AT', 'ATGC', 'GGGGN', 'T']
lengths = list(map(lambda s: len(s), seqs))
filtered = list(filter(lambda s: len(s) >= 3, seqs))
total_len = reduce(lambda acc, s: acc + len(s), seqs, 0)

print(lengths, filtered, total_len)
```
</details>

## 8) Modules and imports

**Task:** Using the `math` module, compute the hypotenuse for sides `a=3`, `b=4` in two different import styles:

- `import math` then call the appropriate function
- `from math import hypot` then call the function directly


In [ ]:
# Your attempt:
# import math
# # TODO: use math.???
#
# from math import hypot
# # TODO: call hypot directly


<details><summary><strong>View answer</strong></summary>

```python
import math
h1 = math.hypot(3, 4)

from math import hypot
h2 = hypot(3, 4)

print(h1, h2)
```
</details>

## 9) Regular expressions — N-glycosylation motif

The consensus motif is `N[^P][ST]`: **N**, then **not P**, then **S or T**.

**Task:** Load sequences from `example.fasta` (ignore header lines starting with `>`), join them per record, and report **0-based indices** of all motif matches for each sequence.


In [ ]:
# Your attempt:
# import re
# def read_fasta(path):
#     # TODO: return dict{name:sequence}
#     pass
#
# # TODO: find motif positions with re.finditer


<details><summary><strong>View answer</strong></summary>

```python
import re

def read_fasta(path):
    records = {}
    name = None
    seq_parts = []
    with open(path, 'r') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            if line.startswith('>'):
                if name is not None:
                    records[name] = ''.join(seq_parts)
                name = line[1:].strip()
                seq_parts = []
            else:
                seq_parts.append(line)
    if name is not None:
        records[name] = ''.join(seq_parts)
    return records

motif = re.compile(r'N[^P][ST]')
records = read_fasta('example.fasta')
for name, seq in records.items():
    positions = [m.start() for m in motif.finditer(seq)]
    print(name, positions)
```
</details>

## 10) `os` basics — remove a file vs remove a directory

**Task:** Create a temporary file and directory, then remove the file with the correct function and the directory with the correct function (they are **different**!). *Do not* delete anything outside this notebook folder.


In [ ]:
# Your attempt:
# import os, shutil
# # TODO: create tmp file and dir, then remove using appropriate functions


<details><summary><strong>View answer</strong></summary>

```python
import os, shutil, tempfile

# create a temp file
tmp_file = Path('to_delete.txt')
tmp_file.write_text('bye')

# create a temp directory
tmp_dir = Path('to_delete_dir')
tmp_dir.mkdir(exist_ok=True)

# remove file
os.remove(tmp_file)

# remove (empty) directory
os.rmdir(tmp_dir)

print('Removed file and directory')
```
</details>

## 11) Simulated CLI parsing (`sys.argv`)

**Task:** Implement `parse_args(argv)` that recognizes `--in <path>` and `--min-len <int>` and returns a dict.  
Then call it with a simulated argv list.


In [ ]:
# Your attempt:
# import sys
# def parse_args(argv):
#     # TODO
#     return {}
#
# args = parse_args(['script.py', '--in', 'example.fasta', '--min-len', '5'])
# args


<details><summary><strong>View answer</strong></summary>

```python
def parse_args(argv):
    args = {'in': None, 'min_len': None}
    i = 1  # skip script name
    while i < len(argv):
        if argv[i] == '--in':
            args['in'] = argv[i+1]
            i += 2
        elif argv[i] == '--min-len':
            args['min_len'] = int(argv[i+1])
            i += 2
        else:
            raise SystemExit(f'Unknown argument: {argv[i]}')
    return args

parse_args(['script.py', '--in', 'example.fasta', '--min-len', '5'])
```
</details>

---

✅ That’s it! If you want, extend any exercise (e.g., add error handling, doctests, or unit tests).